# Can we predict whether a student passes the closing quiz?
Protocol: `protocol.yaml`. Target `passed_quiz`; group holdout and 5-fold group CV on `student_key`; model chosen by mean CV ROC AUC; holdout scored once.

**Prediction-time judgement.** `enrol_year`, `school_code`, `prior_score` (placement test) are known before the session. `hints_used` accrues during the session, before the closing quiz. `session_minutes` is the session length; I assume it is known when the quiz is taken (it is not evidently recorded after the outcome). If the prediction must be made before the session starts, only the first three are valid — this is an assumption the reader should confirm. `student_key` is excluded (entity key).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import yaml
proto = yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest() == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
hold_rows = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==proto['dataset']['rows'] and len(hold_rows)==proto['split']['holdout_rows']
train_rows = np.setdiff1d(np.arange(len(df)), hold_rows)
train, hold = df.iloc[train_rows], df.iloc[hold_rows]
assert not set(train.student_key) & set(hold.student_key)
print(len(train), len(hold), train.passed_quiz.mean())

## Features and candidates (all named before any scoring)

In [ ]:
TARGET='passed_quiz'
FEATURES=['enrol_year','school_code','prior_score','session_minutes','hints_used']
# school_code is a code, not a quantity; kept numeric for simplicity
candidates = {
 'baseline_prior': lambda: DummyClassifier(strategy='prior'),
 'logreg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
 'logreg_C0.05': lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=1000)),
 'hgb_shallow': lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
 'hgb_deeper': lambda: HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=200, min_samples_leaf=50, random_state=0),
}
print(list(candidates))

## Cross-validation on training rows only (preprocessing is inside each pipeline, fitted per fold)

In [ ]:
X, y, g = train[FEATURES], train[TARGET].to_numpy(), train.student_key
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X, y, g))
cv_scores = {}
for name, mk in candidates.items():
    s=[]
    for tr, va in folds:
        m = mk().fit(X.iloc[tr], y[tr])
        s.append(float(roc_auc_score(y[va], m.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(cv.sort_values('mean', ascending=False))
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)

## Refit chosen candidate on all training rows; score once on the holdout

In [ ]:
final = candidates[chosen]().fit(X, y)
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('holdout roc_auc', holdout_auc)
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(i) for i in train_rows[tr]], [int(i) for i in train_rows[va]]] for tr,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_rows],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout number is the headline; it was computed once, after selection on CV. The result depends on the prediction-time assumption about `session_minutes` and `hints_used` stated at the top.